# 浙江水利 · 水位数据导出工具

这个 Notebook 负责从 SQLite 数据库导出 CSV，保留 Notebook 使用方式。

运行顺序：

1. 运行“配置与工具函数”单元格。
2. 运行需要的导出函数，例如 `export_all_data()`、`export_valid_water_data()`、`export_by_city()`。
3. CSV 默认写入 `zhejiang_water_data/exports`，编码为 `utf-8-sig`，方便 Excel 直接打开中文。

导出逻辑：

- 如果新表 `water_level_records_v2` 存在且有数据，优先导出新表。
- 如果新表不存在或没有数据，则回退导出旧表 `zhejiang_water_data`。
- 排序时使用数值水位排序，不按字符串排序。

- ????? `rainfall_records_v1` ?????????? CSV?


In [1]:
import csv
import re
import sqlite3
import logging
from pathlib import Path
from typing import List, Tuple

RUNTIME_ROOT = Path(r"E:\\AAAqian\\storm_surge_runtime_data")
DATA_PATH = RUNTIME_ROOT / "zhejiang_water_data"
DB_FILE = "zhejiang_water_data.db"
LOG_FILE = "zhejiang_export.log"
NEW_TABLE = "water_level_records_v2"
LEGACY_TABLE = "zhejiang_water_data"
RAIN_TABLE = "rainfall_records_v1"
EXPORT_DIR = "exports"

DB_PATH = DATA_PATH / DB_FILE
EXPORT_PATH = DATA_PATH / EXPORT_DIR
LOG_PATH = DATA_PATH / "logs" / LOG_FILE

DATA_PATH.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
EXPORT_PATH.mkdir(parents=True, exist_ok=True)

EXPORT_FIELDS = [
    "city",
    "city_county",
    "station_name",
    "station_type",
    "reported_at",
    "water_level",
    "crawl_time",
]

CSV_HEADERS = {
    "city": "地级市",
    "city_county": "市县",
    "station_name": "站名",
    "station_type": "站点类型",
    "reported_at": "上报时间",
    "water_level": "水位_m",
    "crawl_time": "抓取时间",
}

DEDUP_FIELDS = ["city", "city_county", "station_name", "reported_at"]

RAINFALL_EXPORT_FIELDS = [
    "city",
    "city_county",
    "station_name",
    "station_code",
    "period_start",
    "period_end",
    "rainfall",
    "crawl_time",
]

RAINFALL_CSV_HEADERS = {
    "city": "地级市",
    "city_county": "市县",
    "station_name": "站名",
    "station_code": "站码",
    "period_start": "统计开始时间",
    "period_end": "统计结束时间",
    "rainfall": "雨量_mm",
    "crawl_time": "抓取时间",
}
RAINFALL_DEDUP_FIELDS = ["city", "city_county", "station_name", "station_code", "period_start", "period_end"]

logger = logging.getLogger("zhejiang_water_export")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
file_handler = logging.FileHandler(LOG_PATH, encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.INFO)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)


def check_database(db_path: Path = DB_PATH) -> bool:
    if not db_path.exists():
        logger.error("数据库文件不存在: %s，请先运行爬虫采集数据", db_path)
        return False
    return True


def table_exists(conn, table_name: str) -> bool:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table_name,))
    return cursor.fetchone() is not None


def table_count(conn, table_name: str) -> int:
    cursor = conn.cursor()
    cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"')
    return int(cursor.fetchone()[0])


def get_source_table(conn) -> str:
    """优先使用有数据的新表；新表无数据时回退旧表。"""
    if table_exists(conn, NEW_TABLE) and table_count(conn, NEW_TABLE) > 0:
        return NEW_TABLE
    if table_exists(conn, LEGACY_TABLE):
        return LEGACY_TABLE
    raise RuntimeError(f"数据库中没有 {NEW_TABLE} 或 {LEGACY_TABLE} 表")


def get_columns(conn, table_name: str) -> List[str]:
    cursor = conn.cursor()
    cursor.execute(f'PRAGMA table_info("{table_name}")')
    return [row[1] for row in cursor.fetchall()]


def sanitize_filename(value: str) -> str:
    return re.sub(r"[\\/:*?\"<>|\s]+", "_", str(value or "unknown")).strip("_")


def build_dedup_query(table: str, where_clause: str = "", order_clause: str = "") -> str:
    """导出前按 city/city_county/station_name/reported_at 去重，保留 crawl_time 最新记录。"""
    select_fields = ", ".join(f'"{field}"' for field in EXPORT_FIELDS)
    where_sql = f"WHERE {where_clause}" if where_clause else ""
    order_sql = order_clause or "ORDER BY reported_at DESC, city, city_county, station_name"
    return f"""
        WITH ranked AS (
            SELECT
                {select_fields},
                ROW_NUMBER() OVER (
                    PARTITION BY city, city_county, station_name, reported_at
                    ORDER BY crawl_time DESC, id DESC
                ) AS rn
            FROM "{table}"
            {where_sql}
        )
        SELECT {select_fields}
        FROM ranked
        WHERE rn = 1
        {order_sql}
    """


def write_csv(rows: List[sqlite3.Row], output_file: Path, headers: dict = None) -> Path:
    output_file.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        logger.warning("没有可导出的记录: %s", output_file)
        return output_file

    with output_file.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        active_headers = headers or CSV_HEADERS
        writer.writerow([active_headers.get(key, key) for key in rows[0].keys()])
        for row in rows:
            writer.writerow([row[key] for key in row.keys()])

    logger.info("已导出 %s 条记录: %s", len(rows), output_file)
    return output_file


def fetch_rows(sql: str, params: Tuple = (), db_path: Path = DB_PATH) -> List[sqlite3.Row]:
    if not check_database(db_path):
        return []
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        return list(conn.execute(sql, params).fetchall())
    finally:
        conn.close()


def export_all_data(output_file: str = None, db_path: Path = DB_PATH) -> Path:
    if not check_database(db_path):
        return Path("")

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        table = get_source_table(conn)
        output = EXPORT_PATH / (output_file or f"浙江省水位数据_全部_{table}.csv")
        rows = list(conn.execute(build_dedup_query(table)).fetchall())
        return write_csv(rows, output)
    finally:
        conn.close()


def export_valid_water_data(output_file: str = None, db_path: Path = DB_PATH) -> Path:
    if not check_database(db_path):
        return Path("")

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        table = get_source_table(conn)
        output = EXPORT_PATH / (output_file or f"浙江省水位数据_有效水位_{table}.csv")
        rows = list(conn.execute(build_dedup_query(
            table,
            where_clause="water_level IS NOT NULL AND TRIM(CAST(water_level AS TEXT)) != ''",
            order_clause="ORDER BY CAST(water_level AS REAL) DESC",
        )).fetchall())
        return write_csv(rows, output)
    finally:
        conn.close()


# 兼容旧 Notebook 里的函数名。
def export_water_data(output_file: str = None, db_path: Path = DB_PATH) -> Path:
    return export_valid_water_data(output_file=output_file, db_path=db_path)


def export_by_city(group_field: str = "city_county", output_dir: str = None, db_path: Path = DB_PATH) -> List[Path]:
    """按 city 或 city_county 拆分导出。"""
    if not check_database(db_path):
        return []

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        table = get_source_table(conn)
        columns = get_columns(conn, table)
        if group_field not in columns:
            fallback = "city_county" if "city_county" in columns else "city"
            logger.warning("字段 %s 不存在，改用 %s 分组", group_field, fallback)
            group_field = fallback

        target_dir = EXPORT_PATH / (output_dir or f"按{group_field}拆分_{table}")
        target_dir.mkdir(parents=True, exist_ok=True)

        groups = [row[0] for row in conn.execute(f"""
            SELECT DISTINCT "{group_field}"
            FROM "{table}"
            WHERE "{group_field}" IS NOT NULL AND "{group_field}" != ''
            ORDER BY "{group_field}"
        """).fetchall()]

        output_files: List[Path] = []
        for group_value in groups:
            rows = list(conn.execute(build_dedup_query(
                table,
                where_clause=(
                    f'"{group_field}" = ? '
                    "AND water_level IS NOT NULL "
                    "AND TRIM(CAST(water_level AS TEXT)) != ''"
                ),
                order_clause="ORDER BY CAST(water_level AS REAL) DESC",
            ), (group_value,)).fetchall())
            output_file = target_dir / f"{sanitize_filename(group_value)}.csv"
            write_csv(rows, output_file)
            output_files.append(output_file)

        logger.info("按 %s 拆分导出完成，共 %s 个文件", group_field, len(output_files))
        return output_files
    finally:
        conn.close()


def rainfall_table_exists(conn) -> bool:
    return table_exists(conn, RAIN_TABLE) and table_count(conn, RAIN_TABLE) > 0


def build_rainfall_dedup_query(where_clause: str = "", order_clause: str = "") -> str:
    """Deduplicate rainfall by station and period; keep latest crawl_time."""
    select_fields = ", ".join(f'"{field}"' for field in RAINFALL_EXPORT_FIELDS)
    partition_fields = ", ".join(RAINFALL_DEDUP_FIELDS)
    where_sql = f"WHERE {where_clause}" if where_clause else ""
    order_sql = order_clause or "ORDER BY period_start DESC, city, city_county, station_name"
    return f"""
        WITH ranked AS (
            SELECT
                {select_fields},
                ROW_NUMBER() OVER (
                    PARTITION BY {partition_fields}
                    ORDER BY crawl_time DESC, id DESC
                ) AS rn
            FROM "{RAIN_TABLE}"
            {where_sql}
        )
        SELECT {select_fields}
        FROM ranked
        WHERE rn = 1
        {order_sql}
    """


def export_rainfall_data(output_file: str = None, db_path: Path = DB_PATH) -> Path:
    """Export rainfall records. Unit: mm; period_start/period_end define the accumulation window."""
    if not check_database(db_path):
        return Path("")

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        if not rainfall_table_exists(conn):
            logger.warning("Rainfall table missing or empty: %s", RAIN_TABLE)
            return Path("")
        output = EXPORT_PATH / (output_file or f"zhejiang_rainfall_all_{RAIN_TABLE}.csv")
        rows = list(conn.execute(build_rainfall_dedup_query()).fetchall())
        return write_csv(rows, output, headers=RAINFALL_CSV_HEADERS)
    finally:
        conn.close()


def export_valid_rainfall_data(output_file: str = None, db_path: Path = DB_PATH) -> Path:
    """Export records with valid rainfall values."""
    if not check_database(db_path):
        return Path("")

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        if not rainfall_table_exists(conn):
            logger.warning("Rainfall table missing or empty: %s", RAIN_TABLE)
            return Path("")
        output = EXPORT_PATH / (output_file or f"zhejiang_rainfall_valid_{RAIN_TABLE}.csv")
        rows = list(conn.execute(build_rainfall_dedup_query(
            where_clause="rainfall IS NOT NULL AND TRIM(CAST(rainfall AS TEXT)) != ''",
            order_clause="ORDER BY period_start DESC, CAST(rainfall AS REAL) DESC",
        )).fetchall())
        return write_csv(rows, output, headers=RAINFALL_CSV_HEADERS)
    finally:
        conn.close()


def export_rainfall_by_city(group_field: str = "city_county", output_dir: str = None, db_path: Path = DB_PATH) -> List[Path]:
    """Export rainfall by city or city_county."""
    if not check_database(db_path):
        return []

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    try:
        if not rainfall_table_exists(conn):
            logger.warning("Rainfall table missing or empty: %s", RAIN_TABLE)
            return []
        columns = get_columns(conn, RAIN_TABLE)
        if group_field not in columns:
            fallback = "city_county" if "city_county" in columns else "city"
            logger.warning("Field %s missing; use %s for rainfall grouping", group_field, fallback)
            group_field = fallback

        target_dir = EXPORT_PATH / (output_dir or f"rainfall_by_{group_field}_{RAIN_TABLE}")
        target_dir.mkdir(parents=True, exist_ok=True)

        groups = [row[0] for row in conn.execute(f"""
            SELECT DISTINCT "{group_field}"
            FROM "{RAIN_TABLE}"
            WHERE "{group_field}" IS NOT NULL AND "{group_field}" != ''
            ORDER BY "{group_field}"
        """).fetchall()]

        output_files: List[Path] = []
        for group_value in groups:
            rows = list(conn.execute(build_rainfall_dedup_query(
                where_clause=(
                    f'"{group_field}" = ? '
                    "AND rainfall IS NOT NULL "
                    "AND TRIM(CAST(rainfall AS TEXT)) != ''"
                ),
                order_clause="ORDER BY period_start DESC, CAST(rainfall AS REAL) DESC",
            ), (group_value,)).fetchall())
            output_file = target_dir / f"{sanitize_filename(group_value)}.csv"
            write_csv(rows, output_file, headers=RAINFALL_CSV_HEADERS)
            output_files.append(output_file)

        logger.info("Rainfall grouped by %s exported, %s files", group_field, len(output_files))
        return output_files
    finally:
        conn.close()


def show_database_summary(db_path: Path = DB_PATH) -> None:
    if not check_database(db_path):
        return

    conn = sqlite3.connect(db_path)
    try:
        table = get_source_table(conn)
        columns = get_columns(conn, table)
        total = table_count(conn, table)
        valid = conn.execute(f"""
            SELECT COUNT(*)
            FROM "{table}"
            WHERE water_level IS NOT NULL AND TRIM(CAST(water_level AS TEXT)) != ''
        """).fetchone()[0]
        min_max = conn.execute(
            f'SELECT MIN(CAST(water_level AS REAL)), MAX(CAST(water_level AS REAL)) FROM "{table}" WHERE water_level IS NOT NULL'
        ).fetchone()
        logger.info("Current water export table: %s", table)
        logger.info("Water columns: %s", ", ".join(columns))
        logger.info("Water total: %s, valid water level: %s, min/max: %s / %s", total, valid, min_max[0], min_max[1])
        if rainfall_table_exists(conn):
            rain_total = table_count(conn, RAIN_TABLE)
            rain_valid = conn.execute(f"""
                SELECT COUNT(*)
                FROM "{RAIN_TABLE}"
                WHERE rainfall IS NOT NULL AND TRIM(CAST(rainfall AS TEXT)) != ''
            """).fetchone()[0]
            rain_period = conn.execute(f'SELECT MIN(period_start), MAX(period_end) FROM "{RAIN_TABLE}"').fetchone()
            logger.info("Rainfall table: %s, total: %s, valid: %s, period: %s to %s", RAIN_TABLE, rain_total, rain_valid, rain_period[0], rain_period[1])
        else:
            logger.info("Rainfall table %s missing or empty", RAIN_TABLE)

    finally:
        conn.close()


In [2]:
# 一键导出：运行这个单元格会生成 CSV 文件
# 输出目录：zhejiang_water_data/exports

show_database_summary()
export_all_data()
export_valid_water_data()
export_rainfall_data()
export_by_city(group_field="city_county")
# 如果也想按市县拆分雨量，再取消下一行注释
# export_rainfall_by_city(group_field="city_county")
# 如果还想按地级市拆分，再取消下一行注释
# export_by_city(group_field="city")


2026-07-11 01:49:03,126 [INFO] 当前导出表: water_level_records_v2
2026-07-11 01:49:03,127 [INFO] 字段: id, city, city_county, station_name, station_code, station_type, order_no, time_str, reported_at, water_level, raw_water_level, area_type, crawl_time, source_url, created_at, updated_at
2026-07-11 01:49:03,127 [INFO] 总记录数: 69147，有效水位: 69147，最小/最大水位: -17.85 / 1090.2
2026-07-11 01:49:04,185 [INFO] 已导出 69147 条记录: zhejiang_water_data\exports\浙江省水位数据_全部_water_level_records_v2.csv
2026-07-11 01:49:05,259 [INFO] 已导出 69147 条记录: zhejiang_water_data\exports\浙江省水位数据_有效水位_water_level_records_v2.csv
2026-07-11 01:49:05,337 [INFO] 已导出 36 条记录: zhejiang_water_data\exports\按city_county拆分_water_level_records_v2\丽-.csv
2026-07-11 01:49:05,363 [INFO] 已导出 271 条记录: zhejiang_water_data\exports\按city_county拆分_water_level_records_v2\丽-云和.csv
2026-07-11 01:49:05,393 [INFO] 已导出 509 条记录: zhejiang_water_data\exports\按city_county拆分_water_level_records_v2\丽-庆元.csv
2026-07-11 01:49:05,425 [INFO] 已导出 712 条记录: zhejiang_water

[WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-云和.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-庆元.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-景宁.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-松阳.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-缙云.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-莲都.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-遂昌.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-青田.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/丽-龙泉.csv'),
 WindowsPath('zhejiang_water_data/exports/按city_county拆分_water_level_records_v2/台-